In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: d:\fraud-detection-explainable-ai
src exists: True


In [2]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from src.modeling import train_logistic_regression

In [3]:
PROCESSED_DIR = Path("../data/processed")

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "validation.parquet")

preprocessor = joblib.load(
    PROCESSED_DIR / "preprocessor.joblib"
)

In [4]:
from src.preprocessing import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train = train_df[FEATURES]
y_train = train_df["isFraud"]

X_val = val_df[FEATURES]
y_val = val_df["isFraud"]

In [5]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)

Train: (3820599, 11)
Validation: (1248498, 11)


In [6]:
lr_model = train_logistic_regression(
    X_train_processed,
    y_train
)

In [7]:
lr_val_proba = lr_model.predict_proba(
    X_val_processed
)[:, 1]

pr_auc = average_precision_score(y_val, lr_val_proba)
roc_auc = roc_auc_score(y_val, lr_val_proba)

print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print("Iterations:", lr_model.n_iter_)

PR-AUC: 0.5135
ROC-AUC: 0.9917
Iterations: [391]


In [9]:
import joblib

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(
    lr_model,
    MODELS_DIR / "logistic_regression.joblib"
)

['d:\\fraud-detection-explainable-ai\\models\\logistic_regression.joblib']